# Real-Time Face Mask Detector

This notebook implements **Phase 2** of the Face Mask Detection pipeline:
1. **Face Detection**: Localizes human faces using OpenCV DNN with a Caffe ResNet-10 SSD detector (`deploy.prototxt` + `res10_300x300_ssd_iter_140000.caffemodel`).
2. **Mask Classification**: Extracts each detected face ROI, converts to RGB, resizes to 224x224, and passes it through the trained **MobileNetV2** model (`mask_detector.model.h5`).
3. **Real-Time Display**: Draws bounding boxes (Green for "Mask", Red for "No Mask") along with confidence percentages on live video stream or static images.

In [6]:
# 1. Import necessary packages
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from imutils.video import VideoStream
import numpy as np
import imutils
import time
import cv2
import os
import matplotlib.pyplot as plt

In [7]:
# 2. Function to detect faces and predict mask status
def detect_and_predict_mask(frame, faceNet, maskNet, min_confidence=0.5):
    # Grab the dimensions of the frame and construct a blob for the Caffe face detector (300x300)
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))

    # Pass the blob through the face detection network
    faceNet.setInput(blob)
    detections = faceNet.forward()

    # Initialize our list of faces, their corresponding locations, and mask predictions
    faces = []
    locs = []
    preds = []

    # Loop over all detections (detections.shape[2] is the number of detected objects)
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        # Filter out weak detections by ensuring confidence is greater than threshold
        if confidence > min_confidence:
            # Compute the (x, y)-coordinates of the bounding box
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            # Ensure the bounding boxes fall within the dimensions of the frame
            (startX, startY) = (max(0, startX), max(0, startY))
            (endX, endY) = (min(w - 1, endX), min(h - 1, endY))

            # Extract the face ROI, convert from BGR to RGB, resize to 224x224, and preprocess
            face = frame[startY:endY, startX:endX]
            if face.shape[0] > 0 and face.shape[1] > 0:
                face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                face = cv2.resize(face, (224, 224))
                face = img_to_array(face)
                face = preprocess_input(face)

                # Add the face and bounding box to their respective lists
                faces.append(face)
                locs.append((startX, startY, endX, endY))

    # Only make predictions if at least one face was detected
    if len(faces) > 0:
        faces = np.array(faces, dtype="float32")
        preds = maskNet.predict(faces, batch_size=32)

    # Return a 2-tuple of face locations and predictions
    return (locs, preds)

In [ ]:
#  Load our serialized face detector model from disk
prototxtPath = r"face_detector\deploy.prototxt"
weightsPath = r"face_detector\res10_300x300_ssd_iter_140000.caffemodel"
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

# Load the face mask detector model from disk
maskNet = load_model("mask_detector.model.h5")

In [ ]:
# Inetialize the video stream 
print("[INFO] starting video stream.....")
vs = VideoStream(src=0).start()

# Inetialize the video stream
print("[INFO] starting video stream.....")
vs = VideoStream(src=0).start()#Loop over the frames from the video stream

while True:
    # grab the frames from the threaded video stream and resize it to have maximux width of 400 pixels
    frame = vs.read()
    frame = imutils.resize(frame, width= 400)

    # Detect faces in the frame and deteremine if they are wearing a face mask or not
    (locs, preds) = detect_and_predict_mask(frame, faceNet, maskNet)

    # Loop over the dedicated face locatins and there corresponing locatins 
    for(box, pred) in zip(locs, pred):
        # unpack the bounding box and predictions
        ( startX, startY, endX, endY) = box 
        (mask, withoutMask) = pred

        # determine the class label and color we'll use to drow the bunding box and text 
        label = "Mask"  if mask is > withoutMask else " No Mask "
        color = (0,255,0) if label == "Mask" else (0,0,255)
        

In [ ]:
# # 3. Load our serialized face detector model and trained mask detector model from disk
# prototxtPath = r"face_detector\deploy.prototxt"
# weightsPath = r"face_detector\res10_300x300_ssd_iter_140000.caffemodel"
# faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

# # # Auto-fallback if files are in current working directory
# # if not os.path.exists(prototxtPath):
# #     prototxtPath = "deploy.prototxt"
# # if not os.path.exists(weightsPath):
# #     weightsPath = "res10_300x300_ssd_iter_140000.caffemodel"

# # print(f"[INFO] Loading face detector from: {prototxtPath}, {weightsPath}")
# # faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)
# # print("[INFO] Face detector model loaded successfully!")

# # maskModelPath = "mask_detector.model.h5"
# # print(f"[INFO] Loading face mask detector from: {maskModelPath}")
# # maskNet = load_model(maskModelPath)
# # print("[INFO] Face mask detector model loaded successfully!")

In [ ]:
# 4. Test mask detector on a static image (works without camera)
# def test_on_image(image_path):
#     image = cv2.imread(image_path)
#     if image is None:
#         print(f"[ERROR] Could not read image: {image_path}")
#         return
    
#     orig = image.copy()
#     (locs, preds) = detect_and_predict_mask(image, faceNet, maskNet)
#     print(f"[INFO] Detected faces: {len(locs)}")

#     for (box, pred) in zip(locs, preds):
#         (startX, startY, endX, endY) = box
#         (mask, withoutMask) = pred

#         label = "Mask" if mask > withoutMask else "No Mask"
#         color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
#         confidence = max(mask, withoutMask) * 100
#         labelText = f"{label}: {confidence:.2f}%"

#         cv2.putText(orig, labelText, (startX, startY - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
#         cv2.rectangle(orig, (startX, startY), (endX, endY), color, 2)

#     # Display image inline using Matplotlib
#     orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
#     plt.figure(figsize=(7, 7))
#     plt.imshow(orig_rgb)
#     plt.axis("off")
#     plt.title(f"Mask Detection: {os.path.basename(image_path)}")
#     plt.show()

# # Test with a sample image from dataset
# sample_image = os.path.join("dataset", "with_mask", "img-1.jpeg")
# if os.path.exists(sample_image):
#     test_on_image(sample_image)

In [ ]:
# # 5. Real-Time Webcam / Video Stream Mask Detection
# print("[INFO] Starting video stream (press 'q' in the window to quit)...")
# vs = VideoStream(src=0).start()
# time.sleep(2.0)  # Camera warm up

# try:
#     while True:
#         # Grab frame from video stream and resize to maximum width of 800px
#         frame = vs.read()
#         if frame is None:
#             print("[WARNING] Unable to read frame from webcam.")
#             break
#         frame = imutils.resize(frame, width=800)

#         # Detect faces and predict mask status
#         (locs, preds) = detect_and_predict_mask(frame, faceNet, maskNet)

#         # Loop over the detected face locations and corresponding predictions
#         for (box, pred) in zip(locs, preds):
#             (startX, startY, endX, endY) = box
#             (mask, withoutMask) = pred

#             # Determine the class label and color
#             label = "Mask" if mask > withoutMask else "No Mask"
#             color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
#             confidence = max(mask, withoutMask) * 100
#             labelText = f"{label}: {confidence:.2f}%"

#             # Draw bounding box and text on the frame
#             cv2.putText(frame, labelText, (startX, startY - 10),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
#             cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)

#         # Display the live output frame
#         cv2.imshow("Face Mask Detector - Press Q to Quit", frame)
#         key = cv2.waitKey(1) & 0xFF

#         # If the 'q' key was pressed, break from the loop
#         if key == ord("q"):
#             break
# finally:
#     # Clean up and release webcam
#     print("[INFO] Releasing camera and destroying all windows...")
#     cv2.destroyAllWindows()
#     vs.stop()

[INFO] Starting video stream (press 'q' in the window to quit)...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step
[INFO] Releasing camera and destroying all windows...


KeyboardInterrupt: 